# Imports and basic setup

In [1]:
import os
import glob
import json
import math

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


# Paths & configuration

In [20]:
BASE_DIR = "/users/"
DATA_BASE = f"{BASE_DIR}/preprocessed-dataset"
MODELS_BASE = f"{BASE_DIR}/Models"
FED_CLUSTER_DIR = f"{MODELS_BASE}/FedClustered"

os.makedirs(MODELS_BASE, exist_ok=True)
os.makedirs(FED_CLUSTER_DIR, exist_ok=True)

def auto_find_dataset(folder_prefix: str) -> str:
    matches = glob.glob(f"{DATA_BASE}/{folder_prefix}*")
    if len(matches) == 0:
        raise ValueError(f"No folder found for prefix: {folder_prefix}")
    return matches[0]

DATASET_PATHS = {
    "CSE_CIC_IDS2018": "/users/",
    "CIC_IIoT_2025": "/users/",
    "CIC_BCCC_IoMT_2024": "/users/",
    "Combined": "/users/"
}


DATASET_PATHS

{'CSE_CIC_IDS2018': '/users/',
 'CIC_IIoT_2025': '/users/',
 'CIC_BCCC_IoMT_2024': '/users/',
 'Combined': '/users/'}

# FALCON-ID local model builder

In [21]:
def build_falcon_id_local_model(
    input_dim: int,
    num_classes: int,
    learning_rate: float = 1e-3,
    l2_reg: float = 1e-4,
    dropout_rate: float = 0.4,
) -> tf.keras.Model:
    inp = layers.Input(shape=(input_dim,), name="latent_input")
    x = layers.Reshape((input_dim, 1))(inp)  # (batch, 64, 1)

    # ---- Residual Block 1 ----
    shortcut = layers.Conv1D(
        64, 1, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    x1 = layers.Conv1D(
        64, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Activation("relu")(x1)
    x1 = layers.Conv1D(
        64, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x1)
    x1 = layers.BatchNormalization()(x1)

    x = layers.Add()([shortcut, x1])
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # ---- Residual Block 2 ----
    shortcut2 = layers.Conv1D(
        128, 1, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    x2 = layers.Conv1D(
        128, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.Activation("relu")(x2)
    x2 = layers.Conv1D(
        128, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x2)
    x2 = layers.BatchNormalization()(x2)

    x = layers.Add()([shortcut2, x2])
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.SpatialDropout1D(0.2)(x)

    # ---- BiLSTM ----
    x = layers.Bidirectional(
        layers.LSTM(
            128, return_sequences=True,
            kernel_regularizer=regularizers.l2(l2_reg)
        )
    )(x)

    # ---- Additive Attention (note: sequence length dimension) ----
    score = layers.Dense(128, activation="tanh")(x)
    attn_weights = layers.Dense(1, activation="softmax")(score)
    context = layers.Lambda(
        lambda z: tf.reduce_sum(z[0] * z[1], axis=1)
    )([x, attn_weights])

    # ---- Classifier ----
    x = layers.Dense(
        256, activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(context)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    out = layers.Dense(num_classes, activation="softmax", name="logits")(x)

    model = models.Model(
        inputs=inp,
        outputs=out,
        name="FALCON_ID_CNN_BiLSTM_Attn"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model

# Quick sanity check
tmp_model = build_falcon_id_local_model(input_dim=64, num_classes=5)
tmp_model.summary()

Model: "FALCON_ID_CNN_BiLSTM_Attn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ latent_input        │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 64, 1)     │          0 │ latent_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 64, 64)    │        256 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_7[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 64, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 64, 64)    │     12,352 │ activation_4[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 64, 64)    │        128 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 64, 64)    │          0 │ conv1d_6[0][0],   │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 64, 64)    │          0 │ add_2[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 32, 64)    │          0 │ activation_5[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 32, 128)   │     24,704 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 128)   │        512 │ conv1d_10[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 32, 128)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 32, 128)   │     49,280 │ activation_6[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 32, 128)   │      8,320 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 128)   │        512 │ conv1d_11[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 32, 128)   │          0 │ conv1d_9[0][0],   │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 32, 128)   │          0 │ add_3[0][0]       │
│ (Activation)        │                   │            │                 

 Total params: 493,638 (1.88 MB)

 Trainable params: 492,102 (1.88 MB)

 Non-trainable params: 1,536 (6.00 KB)

# Dataset loader + federated client splitter

In [34]:
def load_preprocessed_dataset(ds_name):
    path = DATASET_PATHS[ds_name]

    # Case 1: Standard datasets (CIC_IIoT_2025, CSE_CIC_IDS2018, IoMT_2024)
    latent_files = {
        "X_train": "train_latent.npy",
        "X_val":   "val_latent.npy",
        "X_test":  "test_latent.npy",
        "y_train": "y_train.npy",
        "y_val":   "y_val.npy",
        "y_test":  "y_test.npy",
    }

    # Case 2: Combined dataset uses "combined_*" prefix
    if ds_name == "Combined":
        latent_files = {
            "X_train": "combined_train_latent.npy",
            "X_val":   "combined_val_latent.npy",
            "X_test":  "combined_test_latent.npy",
            "y_train": "combined_y_train.npy",
            "y_val":   "combined_y_val.npy",
            "y_test":  "combined_y_test.npy",
        }

    # Load all arrays
    X_train = np.load(os.path.join(path, latent_files["X_train"]))
    X_val   = np.load(os.path.join(path, latent_files["X_val"]))
    X_test  = np.load(os.path.join(path, latent_files["X_test"]))

    y_train = np.load(os.path.join(path, latent_files["y_train"]))
    y_val   = np.load(os.path.join(path, latent_files["y_val"]))
    y_test  = np.load(os.path.join(path, latent_files["y_test"]))

    num_classes = len(np.unique(y_train))

    print(f"""📂 Loaded dataset: {ds_name}
  Path: {path}
  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}
  y_train: {y_train.shape}, y_val: {y_val.shape}, y_test: {y_test.shape}
  Num classes: {num_classes}
""")

    return X_train, y_train, X_val, y_val, X_test, y_test, num_classes



def make_federated_clients(
    X_train: np.ndarray,
    y_train: np.ndarray,
    num_clients: int = 10,
    shuffle: bool = True,
):
    n_samples = len(X_train)
    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.default_rng(SEED)
        rng.shuffle(indices)

    X_train = X_train[indices]
    y_train = y_train[indices]

    base_size = n_samples // num_clients
    remainder = n_samples % num_clients

    client_splits = []
    start = 0
    for i in range(num_clients):
        size = base_size + (1 if i < remainder else 0)
        end = start + size
        X_c = X_train[start:end]
        y_c = y_train[start:end]
        client_splits.append((X_c, y_c))
        print(f"  -> Client {i}: {len(X_c)} samples")
        start = end

    return client_splits

# Helper: class distribution & weight-update feature vectors

In [23]:
MAX_SAMPLES_FOR_CLUSTER_STATS = 20000
ALPHA_WEIGHT = 0.7

def get_class_distribution(y: np.ndarray, num_classes: int) -> np.ndarray:
    counts = np.bincount(y, minlength=num_classes).astype("float32")
    total = counts.sum()
    if total == 0:
        return np.ones(num_classes, dtype="float32") / num_classes
    return counts / total

In [24]:
def compute_weight_feature_vector(
    base_weights, client_weights
) -> np.ndarray:
    feats = []
    for w0, w1 in zip(base_weights, client_weights):
        delta = w1 - w0
        delta = delta.astype("float32")
        mean_abs = np.mean(np.abs(delta))
        std = np.std(delta)
        l2 = np.sqrt(np.sum(delta ** 2))
        feats.extend([mean_abs, std, l2])
    return np.array(feats, dtype="float32")

In [25]:
def build_client_feature_matrix(
    base_model: tf.keras.Model,
    client_splits,
    num_classes: int,
    alpha_weight: float = ALPHA_WEIGHT,
    max_samples_per_client: int = MAX_SAMPLES_FOR_CLUSTER_STATS,
    local_epochs: int = 1,
    batch_size: int = 512,
):
    base_weights = base_model.get_weights()
    feature_list = []
    label_dist_list = []

    for idx, (X_c, y_c) in enumerate(client_splits):
        print(f"   🧩 Building features for client {idx}...")
        n_c = len(X_c)
        if n_c == 0:
            print("     -> Client has 0 samples, skipping.")
            # zero-vector placeholder
            weight_feats = np.zeros_like(compute_weight_feature_vector(base_weights, base_weights))
            label_dist = np.ones(num_classes, dtype="float32") / num_classes
        else:
            # Sample a subset of client data for faster local update
            if n_c > max_samples_per_client:
                rng = np.random.default_rng(SEED)
                chosen_idx = rng.choice(n_c, size=max_samples_per_client, replace=False)
                X_sub = X_c[chosen_idx]
                y_sub = y_c[chosen_idx]
            else:
                X_sub = X_c
                y_sub = y_c

            # Build & initialize local model
            local_model = build_falcon_id_local_model(
                input_dim=X_c.shape[1],
                num_classes=num_classes
            )
            local_model.set_weights(base_weights)

            # Local training (1 epoch; only for feature extraction)
            local_model.fit(
                X_sub, y_sub,
                batch_size=batch_size,
                epochs=local_epochs,
                verbose=0,
            )

            client_weights = local_model.get_weights()
            weight_feats = compute_weight_feature_vector(base_weights, client_weights)
            label_dist = get_class_distribution(y_c, num_classes)

        feature_list.append(weight_feats)
        label_dist_list.append(label_dist)

    # Pad weight feature vectors to same length if needed (due to any weird shapes)
    max_len = max(len(f) for f in feature_list)
    padded_feats = []
    for f in feature_list:
        if len(f) < max_len:
            pad = np.zeros(max_len - len(f), dtype="float32")
            f = np.concatenate([f, pad], axis=0)
        padded_feats.append(f)

    weight_feats_mat = np.stack(padded_feats, axis=0)  # (num_clients, feat_dim)
    label_dist_mat = np.stack(label_dist_list, axis=0)  # (num_clients, num_classes)

    weight_feats_scaled = alpha_weight * weight_feats_mat
    label_feats_scaled = (1.0 - alpha_weight) * label_dist_mat

    client_features = np.concatenate(
        [weight_feats_scaled, label_feats_scaled], axis=1
    )

    print(f"   ✅ client_features shape: {client_features.shape}")
    return client_features, weight_feats_mat, label_dist_mat

# Clustering: KMeans on hybrid feature vectors

In [26]:
def cluster_clients_from_features(
    client_features: np.ndarray,
    min_clusters: int = 2,
    max_clusters: int = 3,
):
    num_clients = client_features.shape[0]

    if num_clients < 4:
        n_clusters = 1
    elif num_clients < 6:
        n_clusters = 2
    else:
        n_clusters = max_clusters

    print(f"   🔢 num_clients={num_clients}, using n_clusters={n_clusters}")

    if n_clusters == 1:
        labels = np.zeros(num_clients, dtype="int32")
        return labels, n_clusters

    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(client_features)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=SEED,
        n_init=10
    )
    labels = kmeans.fit_predict(feats_scaled)
    return labels, n_clusters

# Train cluster-specific models & evaluate

In [27]:
CLUSTER_TRAIN_MAX_SAMPLES = 500_000
CLUSTER_EPOCHS = 3
CLUSTER_BATCH_SIZE = 1024

In [28]:
def train_model_on_subset(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    num_classes: int,
    epochs: int = CLUSTER_EPOCHS,
    batch_size: int = CLUSTER_BATCH_SIZE,
):
    input_dim = X_train.shape[1]
    model = build_falcon_id_local_model(
        input_dim=input_dim,
        num_classes=num_classes
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1,
    )
    return model, history

In [29]:
def evaluate_model(
    model: tf.keras.Model,
    X_test: np.ndarray,
    y_test: np.ndarray,
):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    )

    return loss, acc, cm, report

In [30]:
def train_cluster_models_for_dataset(
    dataset_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    num_classes: int,
    client_splits,
    cluster_labels,
    output_root: str = FED_CLUSTER_DIR,
):
    ds_dir = os.path.join(output_root, dataset_name)
    os.makedirs(ds_dir, exist_ok=True)

    num_clients = len(client_splits)
    cluster_labels = np.asarray(cluster_labels)
    cluster_ids = sorted(np.unique(cluster_labels))

    # Save cluster assignments + per-client stats
    assignments = {}
    for i in range(num_clients):
        X_c, y_c = client_splits[i]
        assignments[f"client_{i}"] = {
            "cluster_id": int(cluster_labels[i]),
            "num_samples": int(len(X_c)),
            "class_distribution": get_class_distribution(y_c, num_classes).tolist(),
        }

    with open(os.path.join(ds_dir, "cluster_assignments.json"), "w") as f:
        json.dump(assignments, f, indent=2)

    print(f"   💾 Saved cluster_assignments.json at {ds_dir}")

    summary_rows = []

    for cid in cluster_ids:
        print(f"\n   🚀 Training model for cluster {cid} (dataset: {dataset_name})")
        # Collect all client data in this cluster
        X_list = []
        y_list = []
        client_indices = np.where(cluster_labels == cid)[0]

        for idx in client_indices:
            X_c, y_c = client_splits[idx]
            if len(X_c) > 0:
                X_list.append(X_c)
                y_list.append(y_c)

        if not X_list:
            print(f"   ⚠ Cluster {cid} has no samples. Skipping.")
            continue

        X_cluster = np.concatenate(X_list, axis=0)
        y_cluster = np.concatenate(y_list, axis=0)
        n_cluster = len(X_cluster)
        print(f"     -> Cluster {cid}: {n_cluster} train samples from {len(client_indices)} clients")

        # Subsample cluster training data if extremely large
        if n_cluster > CLUSTER_TRAIN_MAX_SAMPLES:
            rng = np.random.default_rng(SEED)
            chosen_idx = rng.choice(n_cluster, size=CLUSTER_TRAIN_MAX_SAMPLES, replace=False)
            X_cluster_sub = X_cluster[chosen_idx]
            y_cluster_sub = y_cluster[chosen_idx]
        else:
            X_cluster_sub = X_cluster
            y_cluster_sub = y_cluster

        # For validation we use overall X_val, y_val of dataset
        cluster_model, hist = train_model_on_subset(
            X_cluster_sub, y_cluster_sub,
            X_val, y_val,
            num_classes=num_classes,
        )

        test_loss, test_acc, cm, report = evaluate_model(
            cluster_model,
            X_test,
            y_test
        )

        print(f"     ✅ Cluster {cid} test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

        # Save model
        model_path = os.path.join(ds_dir, f"cluster_model_{cid}.keras")
        cluster_model.save(model_path)
        print(f"     💾 Saved model: {model_path}")

        # Save confusion matrix
        cm_path = os.path.join(ds_dir, f"confusion_matrix_cluster_{cid}.npy")
        np.save(cm_path, cm)

        # Save metrics
        metrics = {
            "cluster_id": int(cid),
            "num_train_samples": int(n_cluster),
            "num_clients_in_cluster": int(len(client_indices)),
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
            "classification_report": report,
        }
        metrics_path = os.path.join(ds_dir, f"metrics_cluster_{cid}.json")
        with open(metrics_path, "w") as f:
            json.dump(metrics, f, indent=2)

        print(f"     💾 Saved metrics: {metrics_path}")
        print(f"     💾 Saved confusion matrix: {cm_path}")

        summary_rows.append({
            "dataset": dataset_name,
            "cluster_id": cid,
            "num_train_samples": n_cluster,
            "num_clients_in_cluster": len(client_indices),
            "test_loss": test_loss,
            "test_accuracy": test_acc,
        })

    # Dataset-level summary CSV
    if summary_rows:
        df_summary = pd.DataFrame(summary_rows)
        summary_csv_path = os.path.join(ds_dir, "summary_clusters.csv")
        df_summary.to_csv(summary_csv_path, index=False)
        print(f"\n   📊 Saved dataset-level cluster summary: {summary_csv_path}")

    return

# Global summary aggregator (across all datasets)

In [31]:
def aggregate_all_dataset_summaries(output_root: str = FED_CLUSTER_DIR):
    rows = []
    for ds_name in DATASET_PATHS.keys():
        ds_dir = os.path.join(output_root, ds_name)
        csv_path = os.path.join(ds_dir, "summary_clusters.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            rows.append(df)
    if not rows:
        print("No dataset summaries found.")
        return None

    df_all = pd.concat(rows, ignore_index=True)
    all_summary_path = os.path.join(output_root, "FedClustered_summary_all_datasets.csv")
    df_all.to_csv(all_summary_path, index=False)
    print(f"📊 Global summary saved at: {all_summary_path}")
    return df_all

# Main: Clustered FL pipeline for all datasets

In [35]:
NUM_CLIENTS = 10

all_dataset_summaries = []

for ds_name in DATASET_PATHS.keys():
    print("\n" + "#" * 80)
    print(f"### 🌐 Clustered Federated Learning for Dataset: {ds_name}")
    print("#" * 80)

    # 1) Load dataset
    X_train, y_train, X_val, y_val, X_test, y_test, num_classes = load_preprocessed_dataset(ds_name)
    input_dim = X_train.shape[1]

    # 2) Create federated clients
    print(f"\n👥 Creating {NUM_CLIENTS} federated clients for {ds_name}...")
    client_splits = make_federated_clients(X_train, y_train, num_clients=NUM_CLIENTS)

    # 3) Build base model (shared initialization)
    print(f"\n🧱 Building base model for hybrid feature extraction ({ds_name})...")
    base_model = build_falcon_id_local_model(input_dim=input_dim, num_classes=num_classes)

    # 4) Build client feature matrix (weight updates + label distributions)
    print(f"\n🔍 Computing client feature matrix (Option D hybrid) for {ds_name}...")
    client_features, weight_feats_mat, label_dist_mat = build_client_feature_matrix(
        base_model,
        client_splits,
        num_classes=num_classes,
        alpha_weight=ALPHA_WEIGHT,
        max_samples_per_client=MAX_SAMPLES_FOR_CLUSTER_STATS,
        local_epochs=1,
        batch_size=512,
    )

    # 5) Cluster clients using KMeans
    print(f"\n📌 Clustering clients for {ds_name}...")
    cluster_labels, n_clusters = cluster_clients_from_features(client_features)
    print(f"   Cluster assignments: {cluster_labels.tolist()}")

    # 6) Train cluster-specific models & save results
    print(f"\n🎓 Training cluster-specific models for {ds_name}...")
    train_cluster_models_for_dataset(
        ds_name,
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        num_classes,
        client_splits,
        cluster_labels,
        output_root=FED_CLUSTER_DIR,
    )

print("\n" + "#" * 80)
print("✅ Completed Clustered Federated Learning (Hybrid Option D) for all datasets.")
print("#" * 80)


################################################################################
### 🌐 Clustered Federated Learning for Dataset: CSE_CIC_IDS2018
################################################################################
📂 Loaded dataset: CSE_CIC_IDS2018
  Path: /users/
  X_train: (6737603, 64), X_val: (1443772, 64), X_test: (1443773, 64)
  y_train: (6737603,), y_val: (1443772,), y_test: (1443773,)
  Num classes: 15


👥 Creating 10 federated clients for CSE_CIC_IDS2018...
  -> Client 0: 673761 samples
  -> Client 1: 673761 samples
  -> Client 2: 673761 samples
  -> Client 3: 673760 samples
  -> Client 4: 673760 samples
  -> Client 5: 673760 samples
  -> Client 6: 673760 samples
  -> Client 7: 673760 samples
  -> Client 8: 673760 samples
  -> Client 9: 673760 samples

🧱 Building base model for hybrid feature extraction (CSE_CIC_IDS2018)...

🔍 Computing client feature matrix (Option D hybrid) for CSE_CIC_IDS2018...
   🧩 Building features for client 0...


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


   🧩 Building features for client 1...
   🧩 Building features for client 2...
   🧩 Building features for client 3...
   🧩 Building features for client 4...
   🧩 Building features for client 5...
   🧩 Building features for client 6...
   🧩 Building features for client 7...
   🧩 Building features for client 8...
   🧩 Building features for client 9...
   ✅ client_features shape: (10, 171)

📌 Clustering clients for CSE_CIC_IDS2018...
   🔢 num_clients=10, using n_clusters=3
   Cluster assignments: [2, 0, 2, 2, 1, 2, 2, 1, 2, 0]

🎓 Training cluster-specific models for CSE_CIC_IDS2018...
   💾 Saved cluster_assignments.json at /users/

   🚀 Training model for cluster 0 (dataset: CSE_CIC_IDS2018)
     -> Cluster 0: 1347521 train samples from 2 clients
Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


489/489 ━━━━━━━━━━━━━━━━━━━━ 27s 41ms/step - accuracy: 0.8548 - loss: 0.7137 - val_accuracy: 0.9664 - val_loss: 0.2060
Epoch 2/3
489/489 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - accuracy: 0.9669 - loss: 0.2086 - val_accuracy: 0.9686 - val_loss: 0.1919
Epoch 3/3
489/489 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - accuracy: 0.9679 - loss: 0.1808 - val_accuracy: 0.9688 - val_loss: 0.1619


...

     ✅ Cluster 1 test_loss=0.1653, test_acc=0.9679
     💾 Saved model: /users/
     💾 Saved metrics: /users/
     💾 Saved confusion matrix: /users/

   🚀 Training model for cluster 2 (dataset: Combined)
     -> Cluster 2: 3654698 train samples from 4 clients
Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


489/489 ━━━━━━━━━━━━━━━━━━━━ 30s 45ms/step - accuracy: 0.8551 - loss: 0.7635 - val_accuracy: 0.9608 - val_loss: 0.2407
Epoch 2/3
489/489 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - accuracy: 0.9643 - loss: 0.2197 - val_accuracy: 0.9642 - val_loss: 0.1930
Epoch 3/3
489/489 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - accuracy: 0.9659 - loss: 0.1891 - val_accuracy: 0.9663 - val_loss: 0.1778


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


     ✅ Cluster 2 test_loss=0.1771, test_acc=0.9664
     💾 Saved model: /users/
     💾 Saved metrics: /users/
     💾 Saved confusion matrix: /users/

   📊 Saved dataset-level cluster summary: /users/

################################################################################
✅ Completed Clustered Federated Learning (Hybrid Option D) for all datasets.
################################################################################


# Build global summary table across all datasets

In [36]:
df_all = aggregate_all_dataset_summaries(output_root=FED_CLUSTER_DIR)
df_all

📊 Global summary saved at: /users/


,dataset,cluster_id,num_train_samples,num_clients_in_cluster,test_loss,test_accuracy
0,CSE_CIC_IDS2018,0,1347521,2,0.161760,0.968839
1,CSE_CIC_IDS2018,1,1347520,2,1.207901,0.783623
2,CSE_CIC_IDS2018,2,4042562,6,0.179030,0.958037
3,CIC_IIoT_2025,0,20597,7,1.633672,0.425785
4,CIC_IIoT_2025,1,5885,2,1.879348,0.335553
5,CIC_IIoT_2025,2,2942,1,3.562188,0.150809
6,CIC_BCCC_IoMT_2024,0,473944,2,0.161231,0.966258
7,CIC_BCCC_IoMT_2024,1,1658803,7,0.160847,0.965703
8,CIC_BCCC_IoMT_2024,2,236972,1,0.203612,0.966124
9,Combined,0,2741023,3,0.167661,0.968348
